# 13 · MQA, GQA, and Multi-Head Latent Attention

Companion to **Chapter 13**. You will implement GQA, catch the
`repeat_interleave` bug, implement MLA, and **verify the absorption identity**
that makes MLA actually work.

In [ ]:
import os, sys, math
import torch
import torch.nn as nn
import torch.nn.functional as F
sys.path.insert(0, os.path.abspath('..'))
torch.manual_seed(0)

## 1 · The grouping bug

`repeat_interleave` vs `repeat`. Both run. Only one matches trained weights.

In [ ]:
kv = torch.arange(3).view(1, 3, 1, 1).float()          # 3 KV heads: 0, 1, 2
rep = 2                                                # 2 query heads per group

print("KV heads:            ", kv.flatten().int().tolist())
print("repeat_interleave(2):", kv.repeat_interleave(rep, dim=1).flatten().int().tolist(),
      " <- CORRECT: contiguous groups")
print("repeat(1,2,1,1):     ", kv.repeat(1, rep, 1, 1).flatten().int().tolist(),
      " <- WRONG: interleaved")

assert kv.repeat_interleave(rep, dim=1).flatten().tolist() == [0,0,1,1,2,2]
assert kv.repeat(1, rep, 1, 1).flatten().tolist() == [0,1,2,0,1,2]
print("\nQuery heads are grouped CONTIGUOUSLY: [q0,q1 | q2,q3 | q4,q5].")
print("So KV head 0 must serve q0 and q1 -> you need [k0,k0,k1,k1,k2,k2].")
print("Get this wrong and Llama weights produce mysteriously mediocre output.")

## 2 · GQA: verify it degrades to MHA and MQA

In [ ]:
class GQA(nn.Module):
    def __init__(self, d_model, n_head, n_kv_head, d_head):
        super().__init__()
        assert n_head % n_kv_head == 0
        self.nh, self.nkv, self.hd = n_head, n_kv_head, d_head
        self.rep = n_head // n_kv_head
        self.q = nn.Linear(d_model, n_head    * d_head, bias=False)
        self.k = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.v = nn.Linear(d_model, n_kv_head * d_head, bias=False)
        self.o = nn.Linear(n_head * d_head, d_model,    bias=False)

    def forward(self, x, return_cache=False):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nh,  self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.nkv, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.nkv, self.hd).transpose(1, 2)
        kx = k.repeat_interleave(self.rep, dim=1)
        vx = v.repeat_interleave(self.rep, dim=1)
        y = F.scaled_dot_product_attention(q, kx, vx, is_causal=True)
        out = self.o(y.transpose(1, 2).reshape(B, T, self.nh * self.hd))
        return (out, (k, v)) if return_cache else out

d_model, nh, hd = 128, 8, 16
x = torch.randn(2, 10, d_model)

print(f"{'n_kv_head':>10} {'name':>6} {'cache elems':>13} {'vs MHA':>9} {'attn params':>12}")
for kvh in [8, 4, 2, 1]:
    m = GQA(d_model, nh, kvh, hd)
    out, (kc, vc) = m(x, return_cache=True)
    assert out.shape == (2, 10, d_model)
    assert kc.shape == (2, kvh, 10, hd)
    elems = kc.numel() + vc.numel()
    ref = 2 * 2 * nh * 10 * hd
    name = "MHA" if kvh == nh else ("MQA" if kvh == 1 else "GQA")
    params = sum(p.numel() for p in m.parameters())
    print(f"{kvh:>10} {name:>6} {elems:>13,} {ref/elems:>8.0f}x {params:>12,}")

print("\nCache shrinks exactly as n_kv_head. Attention FLOPs are UNCHANGED --")
print("all 8 query heads still attend over the full sequence.")

## 3 · MLA — Multi-head Latent Attention

Cache one small latent per token; decompress K and V on the fly.
Plus a **decoupled RoPE** slice, shared across all heads.

In [ ]:
class MLA(nn.Module):
    """Scaled-down DeepSeek-V3 MLA. Real config in the comments."""
    def __init__(self, d_model=512, n_head=8, kv_lora=64, q_lora=128,
                 qk_nope=32, qk_rope=16, v_head=32):
        super().__init__()          # DeepSeek-V3: 7168, 128, 512, 1536, 128, 64, 128
        self.nh, self.nope, self.rope, self.vh = n_head, qk_nope, qk_rope, v_head
        self.kv_lora = kv_lora
        # keys/values: compress to a latent + a SHARED rope key
        self.W_dkv = nn.Linear(d_model, kv_lora, bias=False)
        self.W_kr  = nn.Linear(d_model, qk_rope, bias=False)      # shared, 1 head
        self.W_uk  = nn.Linear(kv_lora, n_head * qk_nope, bias=False)
        self.W_uv  = nn.Linear(kv_lora, n_head * v_head,  bias=False)
        # queries: also low-rank, also split
        self.W_dq  = nn.Linear(d_model, q_lora, bias=False)
        self.W_uq  = nn.Linear(q_lora, n_head * qk_nope, bias=False)
        self.W_qr  = nn.Linear(q_lora, n_head * qk_rope, bias=False)
        self.W_o   = nn.Linear(n_head * v_head, d_model, bias=False)

    def forward(self, x, return_cache=False):
        B, T, _ = x.shape
        c_kv   = self.W_dkv(x)                                   # (B,T,kv_lora) CACHED
        k_rope = self.W_kr(x)                                    # (B,T,qk_rope) CACHED

        k_nope = self.W_uk(c_kv).view(B, T, self.nh, self.nope)
        v      = self.W_uv(c_kv).view(B, T, self.nh, self.vh)

        c_q    = self.W_dq(x)
        q_nope = self.W_uq(c_q).view(B, T, self.nh, self.nope)
        q_rope = self.W_qr(c_q).view(B, T, self.nh, self.rope)

        # concat halves; the rope KEY is broadcast across all heads
        q = torch.cat([q_nope, q_rope], -1).transpose(1, 2)      # (B,nh,T,nope+rope)
        k = torch.cat([k_nope,
                       k_rope[:, :, None].expand(-1, -1, self.nh, -1)],
                      -1).transpose(1, 2)
        v = v.transpose(1, 2)                                    # (B,nh,T,v_head)

        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = self.W_o(y.transpose(1, 2).reshape(B, T, self.nh * self.vh))
        return (out, (c_kv, k_rope)) if return_cache else out


mla = MLA()
x = torch.randn(2, 10, 512)
out, (c_kv, k_rope) = mla(x, return_cache=True)
print(f"output {tuple(out.shape)}")
print(f"cached: c_kv {tuple(c_kv.shape)} + k_rope {tuple(k_rope.shape)}")
per_tok = c_kv.shape[-1] + k_rope.shape[-1]
print(f"        = {per_tok} values per token per layer")
assert out.shape == (2, 10, 512) and per_tok == 64 + 16

## 4 · THE ABSORPTION IDENTITY

This is the engine of MLA, and why the RoPE split has to exist.

$$(W^{UQ}c^Q)^\top(W^{UK}c^{KV}) = (c^Q)^\top\underbrace{(W^{UQ})^\top W^{UK}}_{\text{precompute once}}c^{KV}$$

In [ ]:
H, NOPE, KVL, QL = mla.nh, mla.nope, mla.kv_lora, 128
T = 10
c_q  = mla.W_dq(x)[0]                       # (T, q_lora)
c_kv = mla.W_dkv(x)[0]                      # (T, kv_lora)

# --- DIRECT: decompress both sides, then dot ---
q_nope = (c_q  @ mla.W_uq.weight.T).view(T, H, NOPE)     # (T, H, nope)
k_nope = (c_kv @ mla.W_uk.weight.T).view(T, H, NOPE)
s_direct = torch.einsum('qhd,khd->hqk', q_nope, k_nope)

# --- ABSORBED: fold W_uk into the query side; never decompress the keys ---
Wuq = mla.W_uq.weight.view(H, NOPE, QL)                  # (H, nope, q_lora)
Wuk = mla.W_uk.weight.view(H, NOPE, KVL)                 # (H, nope, kv_lora)
A   = torch.einsum('hdr,hds->hrs', Wuq, Wuk)             # (H, q_lora, kv_lora)
s_absorbed = torch.einsum('qr,hrs,ks->hqk', c_q, A, c_kv)

print(f"direct   {tuple(s_direct.shape)}   absorbed {tuple(s_absorbed.shape)}")
print(f"max abs diff: {(s_direct - s_absorbed).abs().max().item():.2e}")
assert torch.allclose(s_direct, s_absorbed, atol=1e-3)
print("\nIDENTICAL ✓  You can attend DIRECTLY IN LATENT SPACE.")
print("The per-head keys never need to be materialised at all.")
print("\nNow insert RoPE between them: R_{n-m} depends on the QUERY-KEY PAIR,")
print("so it sits inside the product and A can no longer be precomputed.")
print("THAT is why DeepSeek splits off a small RoPE-carrying slice.")

## 5 · The cache comparison, done honestly

Exercise 13.3.

In [ ]:
# DeepSeek-V3 real config
n_head, d_head, n_layer = 128, 128, 61
kv_lora, qk_rope = 512, 64

schemes = {
    "MHA-equivalent": 2 * n_head * d_head,
    "GQA-8":          2 * 8 * d_head,
    "MQA":            2 * 1 * d_head,
    "MLA (actual)":   kv_lora + qk_rope,
}
mha = schemes["MHA-equivalent"]
print(f"{'scheme':>16} {'values/tok/layer':>18} {'vs MHA':>9} {'128k ctx':>11}")
for name, val in schemes.items():
    gb = val * n_layer * 2 * 131072 / 1024**3
    print(f"{name:>16} {val:>18,} {mha/val:>8.1f}x {gb:>10.2f} GB")

print(f"\nMLA vs same-config MHA: {mha/schemes['MLA (actual)']:.1f}x smaller")
print(f"MLA vs GQA-8:           {schemes['GQA-8']/schemes['MLA (actual)']:.1f}x smaller")
print(f"\nMLA caches less than GQA with {2*d_head/ (kv_lora+qk_rope) * 1:.2f}... "
      f"n_kv = {(kv_lora+qk_rope)/(2*d_head):.2f} heads --")
print("yet DeepSeek's ablations rank MLA > GQA > MHA on QUALITY.")
print("\nNOTE: the paper's headline '93.3% reduction' compares against DeepSeek 67B,")
print("not a same-config MHA. Always check what the baseline was.")

## 6 · What it costs on a real fleet

In [ ]:
print("70B-class model, 32k context, 8x H100 (640 GB), 140 GB weights, 80 layers:\n")
avail = 640 - 140 - 40
print(f"cache budget: {avail} GB\n")
print(f"{'scheme':>18} {'GB/user':>9} {'users':>7} {'+fp8 KV':>9}")
for name, per_layer in [("MHA (64 kv heads)", 2*64*128),
                        ("GQA-8",             2*8*128),
                        ("MLA (576)",         576)]:
    gb = per_layer * 80 * 2 * 32768 / 1024**3
    print(f"{name:>18} {gb:>9.1f} {int(avail/gb):>7} {int(avail/(gb/2)):>9}")

print("\nA ~30x difference in users-per-GPU at identical model quality.")
print("Architecture choices show up directly on the invoice -- which is a large")
print("part of why DeepSeek's API pricing undercut competitors so heavily.")

---
## Self-check

1. Does GQA reduce attention FLOPs?
2. Why does MLA need a *decoupled* RoPE dimension?
3. Why is the RoPE **key** shared across heads but the RoPE **query** per-head?
4. You need a 4× cache reduction on an existing Llama model, low risk. What do you do?

<details><summary>Answers</summary>

1. **No.** All query heads still attend over the full sequence. It reduces stored and
   read *bytes* — which is what matters in a memory-bound regime.
2. A position-dependent rotation `R_{n−m}` would sit between `W_UQ` and `W_UK`,
   destroying the absorption identity you verified in §4.
3. The key is **cached**; per-head RoPE keys would cost 128×64 = 8192 values/token —
   14× the entire MLA cache. Queries are recomputed each step and never stored.
4. GQA at `n_kv_head = n_head/4` (needs light uptraining) or fp8 KV quantization
   (needs no retraining). Both low-risk and composable. MQA costs more quality than
   necessary; converting to MLA is a research project.

</details>